# Week 4 Capstone: Retail Sales Analysis\n**Dataset note:** This is a reproducible synthetic retail dataset because the Google Sheets link in the supplied brief could not be retrieved in this environment. It must not be described as actual company data.

## 1. Objective and workflow\nClean and explore sales transactions, analyze business performance, build a sales regression model, and prepare dashboard-ready outputs.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\ndf = pd.read_csv('retail_sales_raw.csv', parse_dates=['Order_Date'])\ndf.head()

In [ ]:
print('Shape:', df.shape)\ndisplay(df.isna().sum().to_frame('Missing values'))\nprint('Duplicate Order IDs:', df.duplicated('Order_ID').sum())\ndisplay(df.describe(include='all').T)

## 2. Data cleaning\nRemove duplicate order IDs, impute missing regions with `Unknown`, impute missing quantity with the median, convert dates and numeric fields, and derive month and profit margin.

In [ ]:
df = df.drop_duplicates(subset=['Order_ID']).copy()\ndf['Region'] = df['Region'].fillna('Unknown')\ndf['Quantity'] = df['Quantity'].fillna(df['Quantity'].median()).astype(int)\nfor c in ['Unit_Price','Discount','Sales','Profit']:\n    df[c] = pd.to_numeric(df[c], errors='coerce')\ndf = df.dropna(subset=['Order_Date','Sales'])\ndf['Month'] = df['Order_Date'].dt.to_period('M').astype(str)\ndf['Profit_Margin'] = np.where(df.Sales != 0, df.Profit/df.Sales, 0)\ndf.info()

## 3. Exploratory data analysis

In [ ]:
display(df[['Sales','Profit','Quantity','Unit_Price','Discount']].describe())\ndisplay(df.groupby('Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order_ID','count')).sort_values('Sales',ascending=False))\ndisplay(df.groupby('Region').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Sales',ascending=False))

In [ ]:
monthly = df.groupby('Month',as_index=False)['Sales'].sum()\nmonthly.plot(x='Month',y='Sales',figsize=(12,4),marker='o',title='Monthly Sales Trend',legend=False)\nplt.xticks(rotation=60); plt.tight_layout(); plt.show()\ndf.groupby('Category')['Sales'].sum().sort_values().plot(kind='barh',title='Sales by Category'); plt.show()

## 4. Regression model to predict sales\nA Random Forest regressor is trained with a held-out 20% test set. Sales is excluded from predictors. Inputs include quantity, unit price, discount and business dimensions.

In [ ]:
features = ['Quantity','Unit_Price','Discount','Region','Category','Sub_Category','Segment','Channel']\nX, y = df[features], df['Sales']\nnum = ['Quantity','Unit_Price','Discount']; cat = [c for c in features if c not in num]\nprep = ColumnTransformer([('num','passthrough',num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])\nmodel = Pipeline([('prep',prep),('rf',RandomForestRegressor(n_estimators=250,random_state=42,min_samples_leaf=2,n_jobs=-1))])\nX_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42)\nmodel.fit(X_train,y_train); pred = model.predict(X_test)\nprint('MAE:',mean_absolute_error(y_test,pred))\nprint('RMSE:',mean_squared_error(y_test,pred)**0.5)\nprint('R2:',r2_score(y_test,pred))\nplt.scatter(y_test,pred,alpha=.5); plt.xlabel('Actual sales'); plt.ylabel('Predicted sales'); plt.title('Actual vs Predicted'); plt.show()

## 5. Export dashboard-ready data\nImport `retail_sales_cleaned.csv` into Power BI. Use the DAX measures and dashboard layout in the accompanying files.

In [ ]:
df.to_csv('retail_sales_cleaned.csv',index=False)\nprint('Cleaned rows:',len(df),'Total sales:',round(df.Sales.sum(),2),'Total profit:',round(df.Profit.sum(),2))